## Packages & Structures

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import os
import wrds


In [ ]:
def find_project_root(start_path, marker="data"):
    path = Path(start_path).resolve()
    for parent in [path] + list(path.parents):
        if (parent / marker).exists():
            return parent
    raise RuntimeError("Project root not found")

PROJECT_ROOT = find_project_root(Path.cwd())
os.chdir(PROJECT_ROOT)

print("Project root:", PROJECT_ROOT)

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
DATA_SAMPLES =  PROJECT_ROOT / "data" / "samples"
DATA_OUTPUT =  PROJECT_ROOT / "data" / "output"

## Datasets

In [ ]:
event_dates = pd.read_csv(DATA_PROCESSED/'event_dates.csv')
crsp = pd.read_csv(DATA_RAW/'crsp_panel.csv')

In [ ]:
crsp.columns

In [ ]:
event_dates.columns

In [ ]:
event_dates['gvkey'] = event_dates['gvkey'].astype('Int64')

In [ ]:
crsp["date"] = pd.to_datetime(crsp["date"])
event_dates["mostimportantdateutc"] = pd.to_datetime(
    event_dates["mostimportantdateutc"]
)

In [ ]:
crsp["log_prev_market_cap"] = np.log(crsp["dlyprevcap"])

In [ ]:
size_controls = crsp[[
    "permno",
    "date",
    "log_prev_market_cap"
]]

event_dates = event_dates.merge(
    size_controls,
    left_on=["permno", "mostimportantdateutc"],
    right_on=["permno", "date"],
    how="left"
)

In [ ]:
event_dates[["permno", "mostimportantdateutc", "log_prev_market_cap"]]

In [ ]:
ratios = pd.read_csv(DATA_RAW / "finratios_ibes.csv")

In [ ]:
ratios

In [ ]:
# clean
event_dates = event_dates.dropna(subset=["permno", "mostimportantdateutc"]).copy()

# same dtype
ratios["permno"] = ratios["permno"].astype("int64")
event_dates["permno"] = event_dates["permno"].astype("int64")

# datetime
ratios["public_date"] = pd.to_datetime(ratios["public_date"])
event_dates["mostimportantdateutc"] = pd.to_datetime(event_dates["mostimportantdateutc"])

# IMPORTANT: sort by date first
ratios = ratios.sort_values(["public_date", "permno"]).reset_index(drop=True)
event_dates = event_dates.sort_values(["mostimportantdateutc", "permno"]).reset_index(drop=True)

ratio_cols = [
    "permno",
    "public_date",
    "bm",
    "roa",
    "roe"
]

event_dates = pd.merge_asof(
    event_dates,
    ratios[ratio_cols],
    left_on="mostimportantdateutc",
    right_on="public_date",
    by="permno",
    direction="backward"
)


In [ ]:
event_dates

In [ ]:
missing_bm = event_dates['bm'].isna().sum()

total = len(event_dates)

print(f"Missing BM: {missing_bm}")
print(f"Percentage: {missing_bm / total:.2%}")

In [ ]:
missing = event_dates[event_dates['bm'].isna()]

missing['public_date'].isna().mean()

In [ ]:
missing = event_dates[event_dates['bm'].isna()]

missing['gvkey'].isin(ratios['gvkey']).mean()

In [ ]:
fundamentals = pd.read_csv(DATA_RAW / "fundamentals.csv")

In [ ]:
fundamentals.head()

In [ ]:
event_dates['gvkey'].isin(fundamentals['gvkey']).mean()

In [ ]:
event_dates = event_dates.sort_values(["gvkey", "date"])
fundamentals = fundamentals.sort_values(["gvkey", "rdq"])

In [ ]:
print(event_dates.dtypes)
print("===============================")
print(fundamentals.dtypes) 

In [ ]:
event_dates = event_dates.dropna(subset=["date"])

In [ ]:

# Dates
event_dates["date"] = pd.to_datetime(event_dates["date"])
fundamentals["rdq"] = pd.to_datetime(fundamentals["rdq"])

# Drop missing merge keys
event_dates_clean = event_dates.dropna(subset=["date", "gvkey"]).copy()
fundamentals_clean = fundamentals.dropna(subset=["rdq", "gvkey", "atq"]).copy()

# Dtypes
event_dates_clean["gvkey"] = event_dates_clean["gvkey"].astype("int64")
fundamentals_clean["gvkey"] = fundamentals_clean["gvkey"].astype("int64")

# Sort
event_dates_clean = event_dates_clean.sort_values(["date", "gvkey"])
fundamentals_clean = fundamentals_clean.sort_values(["rdq", "gvkey"])

# Backward merge
event_dates_merged = pd.merge_asof(
    event_dates_clean,
    fundamentals_clean[
        ["gvkey", "rdq", "atq", "gind"]
    ],
    left_on="date",
    right_on="rdq",
    by="gvkey",
    direction="backward"
)

event_dates_merged["log_assets"] = np.log(event_dates_merged["atq"])

In [ ]:
event_dates_merged = event_dates_merged[['transcriptid', 'companyid', 'companyname', 'mostimportantdateutc',
       'companyname_clean', 'gvkey', 'permno',
       'date', 'bm', 'rdq',
       'atq', 'gind', 'log_assets']].copy()

In [ ]:
word_count = pd.read_csv(DATA_PROCESSED / 'word_count.csv')

In [ ]:
event_dates_merged = event_dates_merged.merge(
    word_count[["transcriptid", "word_count", 'log_word_count']],
    on="transcriptid",
    how="left"
)

In [ ]:
event_dates_merged.to_csv(DATA_PROCESSED / "controls_extensive.csv", index=False)